# PS S6E4 - exp_20260410_039_xgb_formula_logit_min

目的:
 - 039 を既存主力群(025のみ)に加えたとき、025 単体CV,LB超えるかを確認する。

対象:
 1. baseline_025 : 025 単体
 2. set_025_039 : 025 + 039

## Imports

In [1]:
import os
import json
import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedKFold
from sklearn.linear_model import RidgeCV
from sklearn.metrics import balanced_accuracy_score

## CFG

In [2]:
class CFG:
    EXP_ID = "exp_20260410_038_ridge_screen_dao037_vs_core"
    SEED = 42
    N_SPLITS = 5

    TARGET = "Irrigation_Need"
    ID_COL = "id"

    TRAIN_PATH = "/kaggle/input/competitions/playground-series-s6e4/train.csv"
    TEST_PATH = "/kaggle/input/competitions/playground-series-s6e4/test.csv"
    NPY_PATH = "/kaggle/input/datasets/mizushimatoshihiko/npy-files/"

    OUTPUT_DIR = f"/kaggle/working/{EXP_ID}"

    TARGET_MAP = {"Low": 0, "Medium": 1, "High": 2}
    INV_TARGET_MAP = {0: "Low", 1: "Medium", 2: "High"}

    # npy paths
    # OOF_018 = NPY_PATH + "oof_cat_pairwise_te_bias_from_shared_min_proba_biased.npy"
    # PRED_018 = NPY_PATH + "pred_cat_pairwise_te_bias_from_shared_min_proba_biased.npy"

    OOF_025 = NPY_PATH + "oof_lgb_digit_te_min_proba_biased.npy"
    PRED_025 = NPY_PATH + "pred_lgb_digit_te_min_proba_biased.npy"

    # OOF_026 = NPY_PATH + "oof_realmlp_pair_te_min_proba_biased.npy"
    # PRED_026 = NPY_PATH + "pred_realmlp_pair_te_min_proba_biased.npy"

    # OOF_037 = NPY_PATH + "oof_dao_xgb_pair_te_posthoc_only_proba.npy"
    # PRED_037 = NPY_PATH + "pred_dao_xgb_pair_te_posthoc_only_proba.npy"

    OOF_039 = NPY_PATH + "oof_xgb_formula_logit_min_proba.npy"
    PRED_039 = NPY_PATH + "pred_xgb_formula_logit_min_proba.npy"
    
    RIDGE_ALPHAS = np.logspace(-3, 3, 25)

## utility

In [3]:
# ---------------------------------------------------------
# Utils
# ---------------------------------------------------------
def ensure_dir(path):
    os.makedirs(path, exist_ok=True)

def save_json(obj, path):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False, indent=2)

def load_proba(path):
    arr = np.load(path)
    assert arr.ndim == 2 and arr.shape[1] == 3, f"Unexpected shape: {arr.shape} for {path}"
    return arr.astype(np.float32)

def multiclass_proba_to_features(proba_dict, keys):
    """
    Convert multiple (n, 3) probability arrays into one (n, len(keys)*3) feature matrix.
    """
    mats = [proba_dict[k] for k in keys]
    return np.concatenate(mats, axis=1)

def mean_raw_corr(a, b):
    vals = []
    for c in range(a.shape[1]):
        vals.append(np.corrcoef(a[:, c], b[:, c])[0, 1])
    return float(np.mean(vals))

def rankdata_1d(x):
    return pd.Series(x).rank(method="average").to_numpy()

def mean_rank_corr(a, b):
    vals = []
    for c in range(a.shape[1]):
        ra = rankdata_1d(a[:, c])
        rb = rankdata_1d(b[:, c])
        vals.append(np.corrcoef(ra, rb)[0, 1])
    return float(np.mean(vals))

def fit_ridge_multiclass_oof(X, y, X_test, n_splits=5, seed=42, alphas=None):
    """
    Train one RidgeCV per class in OOF manner.
    Inputs:
      X      : (n_train, d)
      y      : (n_train,) class labels 0/1/2
      X_test : (n_test, d)
    Returns:
      oof_pred_proba_like : (n_train, 3)
      test_pred_proba_like: (n_test, 3)
      fold_scores         : list
      chosen_alphas       : list of dicts
      coef_summary        : list of dicts
    """
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)

    oof_pred = np.zeros((len(X), 3), dtype=np.float32)
    test_pred = np.zeros((len(X_test), 3), dtype=np.float32)

    fold_scores = []
    chosen_alphas = []
    coef_summary = []

    for fold, (tr_idx, va_idx) in enumerate(skf.split(X, y), 1):
        X_tr, X_va = X[tr_idx], X[va_idx]
        y_tr, y_va = y[tr_idx], y[va_idx]

        va_pred = np.zeros((len(va_idx), 3), dtype=np.float32)
        te_pred = np.zeros((len(X_test), 3), dtype=np.float32)

        fold_alpha_info = {}
        fold_coef_info = {}

        for cls in range(3):
            y_tr_bin = (y_tr == cls).astype(np.float32)

            model = RidgeCV(alphas=alphas)
            model.fit(X_tr, y_tr_bin)

            va_pred[:, cls] = model.predict(X_va)
            te_pred[:, cls] = model.predict(X_test)

            fold_alpha_info[f"class_{cls}"] = float(model.alpha_)
            fold_coef_info[f"class_{cls}_coef_abs_mean"] = float(np.mean(np.abs(model.coef_)))

        oof_pred[va_idx] = va_pred
        test_pred += te_pred / n_splits

        fold_score = balanced_accuracy_score(y_va, np.argmax(va_pred, axis=1))
        fold_scores.append(float(fold_score))
        chosen_alphas.append(fold_alpha_info)
        coef_summary.append(fold_coef_info)

        print(f"Fold {fold} BA: {fold_score:.6f}")

    return oof_pred, test_pred, fold_scores, chosen_alphas, coef_summary

## load data

In [4]:
# ---------------------------------------------------------
# Load data
# ---------------------------------------------------------
ensure_dir(CFG.OUTPUT_DIR)

train = pd.read_csv(CFG.TRAIN_PATH)
test = pd.read_csv(CFG.TEST_PATH)
y = train[CFG.TARGET].map(CFG.TARGET_MAP).values

proba_oof = {
    # "018": load_proba(CFG.OOF_018),
    "025": load_proba(CFG.OOF_025),
    # "026": load_proba(CFG.OOF_026),
    "039": load_proba(CFG.OOF_039),
}

proba_pred = {
    # "018": load_proba(CFG.PRED_018),
    "025": load_proba(CFG.PRED_025),
    # "026": load_proba(CFG.PRED_026),
    "039": load_proba(CFG.PRED_039),
}

## Screening sets

In [5]:
# ---------------------------------------------------------
# Screening sets
# ---------------------------------------------------------
screen_sets = {
    "baseline_025": ["025"],
    "set_025_039": ["025", "039"],
    # "set_018_025_026": ["018", "025", "026"],
    # "set_018_025_037": ["018", "025", "037"],
    # "set_018_025_026_037": ["018", "025", "026", "037"],

    # optional
    # "set_025_037": ["025", "037"],
    # "set_018_037": ["018", "037"],
}

results = []
coef_rows = []
corr_rows = []

# baseline single-model CVs
for key in ["025", "039"]: # "018", "025", "026", "037"
    score = balanced_accuracy_score(y, np.argmax(proba_oof[key], axis=1))
    results.append({
        "set_name": f"single_{key}",
        "members": key,
        "cv_score": float(score),
        "notes": "single model"
    })

# pairwise raw/rank correlations
base_keys = ["025", "039"]  # "018", "025", "026", "037"
for i in range(len(base_keys)):
    for j in range(i + 1, len(base_keys)):
        a = base_keys[i]
        b = base_keys[j]
        corr_rows.append({
            "a": a,
            "b": b,
            "raw_corr_mean": mean_raw_corr(proba_oof[a], proba_oof[b]),
            "rank_corr_mean": mean_rank_corr(proba_oof[a], proba_oof[b]),
        })

## ridge screening

In [6]:
# ridge screening
for set_name, members in screen_sets.items():
    print(f"\n===== {set_name} =====")
    X = multiclass_proba_to_features(proba_oof, members)
    X_test = multiclass_proba_to_features(proba_pred, members)

    oof_blend, pred_blend, fold_scores, chosen_alphas, coef_summary = fit_ridge_multiclass_oof(
        X=X,
        y=y,
        X_test=X_test,
        n_splits=CFG.N_SPLITS,
        seed=CFG.SEED,
        alphas=CFG.RIDGE_ALPHAS
    )

    cv_score = balanced_accuracy_score(y, np.argmax(oof_blend, axis=1))
    print(f"{set_name} CV: {cv_score:.6f}")

    results.append({
        "set_name": set_name,
        "members": ",".join(members),
        "cv_score": float(cv_score),
        "fold_scores": [float(x) for x in fold_scores],
    })

    # save blend npy per set
    np.save(f"{CFG.OUTPUT_DIR}/oof_{set_name}.npy", oof_blend)
    np.save(f"{CFG.OUTPUT_DIR}/pred_{set_name}.npy", pred_blend)

    # save submission per set
    submission = pd.DataFrame({
        CFG.ID_COL: test[CFG.ID_COL],
        CFG.TARGET: pd.Series(np.argmax(pred_blend, axis=1)).map(CFG.INV_TARGET_MAP)
    })
    submission.to_csv(f"{CFG.OUTPUT_DIR}/submission_{set_name}.csv", index=False)

    # alpha / coef summaries
    for fold_idx, (alpha_info, coef_info) in enumerate(zip(chosen_alphas, coef_summary), 1):
        row = {"set_name": set_name, "fold": fold_idx}
        row.update(alpha_info)
        row.update(coef_info)
        coef_rows.append(row)


===== baseline_025 =====
Fold 1 BA: 0.978330
Fold 2 BA: 0.979055
Fold 3 BA: 0.979795
Fold 4 BA: 0.979691
Fold 5 BA: 0.980021
baseline_025 CV: 0.979378

===== set_025_039 =====
Fold 1 BA: 0.973424
Fold 2 BA: 0.974952
Fold 3 BA: 0.975042
Fold 4 BA: 0.973433
Fold 5 BA: 0.974085
set_025_039 CV: 0.974187


## Save summaries

In [7]:
# ---------------------------------------------------------
# Save summaries
# ---------------------------------------------------------
results_df = pd.DataFrame(results).sort_values("cv_score", ascending=False)
results_df.to_csv(f"{CFG.OUTPUT_DIR}/ridge_screen_results.csv", index=False)

coef_df = pd.DataFrame(coef_rows)
coef_df.to_csv(f"{CFG.OUTPUT_DIR}/ridge_screen_coef_summary.csv", index=False)

corr_df = pd.DataFrame(corr_rows)
corr_df.to_csv(f"{CFG.OUTPUT_DIR}/pairwise_corr_summary.csv", index=False)

summary = {
    "exp_id": CFG.EXP_ID,
    "n_splits": CFG.N_SPLITS,
    "seed": CFG.SEED,
    "alphas": CFG.RIDGE_ALPHAS.tolist(),
    "top_results": results_df.head(10).to_dict(orient="records"),
}
save_json(summary, f"{CFG.OUTPUT_DIR}/cv_result.json")

print("\nTop results:")
print(results_df.head(10))

print("\nSaved files:")
for fn in [
    "ridge_screen_results.csv",
    "ridge_screen_coef_summary.csv",
    "pairwise_corr_summary.csv",
    "cv_result.json",
]:
    print("-", f"{CFG.OUTPUT_DIR}/{fn}")


Top results:
       set_name  members  cv_score         notes  \
0    single_025      025  0.979396  single model   
2  baseline_025      025  0.979378           NaN   
3   set_025_039  025,039  0.974187           NaN   
1    single_039      039  0.971128  single model   

                                         fold_scores  
0                                                NaN  
2  [0.9783298142980573, 0.9790547645808384, 0.979...  
3  [0.9734241579197231, 0.9749519061658561, 0.975...  
1                                                NaN  

Saved files:
- /kaggle/working/exp_20260410_038_ridge_screen_dao037_vs_core/ridge_screen_results.csv
- /kaggle/working/exp_20260410_038_ridge_screen_dao037_vs_core/ridge_screen_coef_summary.csv
- /kaggle/working/exp_20260410_038_ridge_screen_dao037_vs_core/pairwise_corr_summary.csv
- /kaggle/working/exp_20260410_038_ridge_screen_dao037_vs_core/cv_result.json
